In [7]:
import os
# import cohere
from dotenv import load_dotenv
from typing import List
from langchain_cohere import CohereEmbeddings
from llama_index.core import Document as LlamaIndexDoc
from langchain_core.documents import Document as LangChainDoc
from langchain_community.docstore.in_memory import InMemoryDocstore
from llama_index.core.node_parser import SemanticSplitterNodeParser, SentenceSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever

In [8]:
load_dotenv('.env')
COHERE_API_KEY = os.environ['COHERE_API_KEY']

cohere_embed = CohereEmbeddings(
    model="embed-english-v3.0"
)

In [9]:
"""if we are using parent child nodes but think if a child node belongs to some parent node there can be two cases
    1. the child is in somewhere middle of the parent node then when retrieved the parent node will be fine 
    2. but in 2nd case it maybe possible that child is on the edge of parent nodes then it may be possible that the relevant context is either in 
        A. next parent if child is at end edge 
        B. previous parent if child is at strating edge"""

class LlamaSemanticSplitterWrapper(RecursiveCharacterTextSplitter):
    def __init__(self, llama_splitter: SemanticSplitterNodeParser):
        super().__init__()
        self.llama_splitter = llama_splitter

    def split_text(self, text:str) -> List[str]:
        llama_doc = LlamaIndexDoc(text=text)
        nodes = self.llama_splitter.get_nodes_from_documents([llama_doc])
        return [node.get_content() for node in nodes]
    
    def split_documents(self, documents:List[LangChainDoc]) -> List[LangChainDoc]:
        final_docs = []
        for doc in documents:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                final_docs.append(LangChainDoc(page_content=chunk, metadata=doc.metadata.copy()))

        return final_docs

In [10]:
class chunks:
    def __init__(self, embedding=cohere_embed)->None:
        self.embedding = embedding

    def split_text(self,text:str,buffer_size:int=1,breakpoint_percentile_threshold:int=80):

        parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1024,chunks_overlap=150)

        llama_semantic_parser = SemanticSplitterNodeParser(
            embed_model=self.embedding,
            buffer_size=buffer_size,
            breakpoint_percentile_threshold=breakpoint_percentile_threshold)

        child_splitter = LlamaSemanticSplitterWrapper(llama_semantic_parser)

        vector_store = None
        doc_store = InMemoryDocstore()

        retriever = ParentDocumentRetriever(
            vectorstore=vector_store,
            docstore=doc_store,
            child_splitter=child_splitter, # Uses LlamaIndex under the hood
            parent_splitter=parent_splitter
        )
        return retriever

    

In [ ]:
# from langchain_community.vectorstores import Chroma
# from langchain_core.documents import Document as LangChainDoc
# from langchain_community.docstore.in_memory import InMemoryDocstore
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# # Setup standard LangChain storage components
# vectorstore = Chroma(collection_name="semantic_idx", embedding_function=your_lc_embeddings)
# docstore = InMemoryByteStore()
# parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=200)

# # Plug in your custom wrapped LlamaIndex splitter directly as the child splitter!
# retriever = ParentDocumentRetriever(
#     vectorstore=vectorstore,
#     docstore=docstore,
#     child_splitter=langchain_compatible_child_splitter, # Uses LlamaIndex under the hood
#     parent_splitter=parent_splitter
# )

# # Ingest your massive raw string effortlessly
# retriever.add_documents([LangChainDoc(page_content="Your massive text...")])
